# Proyecto SQL

**Objetivos del estudio**

El objetivo de este estudio es explorar la base de datos de un servicio de libros digitales para identificar patrones relevantes sobre publicaciones, editoriales, autores y comportamiento de los usuarios (calificaciones y reseñas).

Los resultados permitirán generar ideas estratégicas con los cuales definir una propuesta de valor para un nuevo producto en el mercado de lectura digital.

In [2]:
# import libraries
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',  # username
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',  # password
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,  # connection port
    'db': 'data-analyst-final-project-db'  # the name of the database
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [3]:
# Exploración de tablas

# función para ejecutar consultas
def run_query(query):
    return pd.read_sql(query, engine)

query_books = "SELECT * FROM books LIMIT 5;"
run_query(query_books)


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [4]:
query_authors = "SELECT * FROM authors LIMIT 5;"
run_query(query_authors)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [5]:
query_publishers = "SELECT * FROM publishers LIMIT 5;"
run_query(query_publishers)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [6]:
query_ratings = "SELECT * FROM ratings LIMIT 5;"
run_query(query_ratings)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [7]:
query_reviews = "SELECT * FROM reviews LIMIT 5;"
run_query(query_reviews)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


Al analizar las tablas podemos concluir preliminarmente que:

El modelo de datos presenta una estructura relacional clara, books actúa como tabla central, authors y publishers se conectan mediante claves foráneas, ratings y reviews permiten medir interacción y percepción de usuarios.

No se identifican inconsistencias estructurales que impidan el análisis, por lo que se puede proceder con las consultas requeridas.

# Ejercicios
**1. Numero de libros publicados despues del 1 de enero de 2000**

In [8]:
query_books_after_2000 = """
SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01';
"""


books_after_2000 = run_query(query_books_after_2000)

books_after_2000

,books_after_2000
0,819


La base de datos contiene 819 libros publicados después del año 2000, lo que indica que una parte significativa del catálogo corresponde a publicaciones relativamente recientes.
Esto sugiere que la plataforma no se limita a títulos clásicos, sino que también incluye literatura contemporánea, lo cual es clave para atraer a usuarios interesados en novedades editoriales.

**2. Número de reseñas y calificación promedio por libro**

In [12]:
query_reviews = """
SELECT 
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS num_reviews,
    AVG(rt.rating) AS avg_rating
FROM books b
LEFT JOIN reviews r ON b.book_id = r.book_id
LEFT JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY b.book_id, b.title
ORDER BY num_reviews DESC;
"""

reviews_stats = run_query(query_reviews)

reviews_stats.head()

,book_id,title,num_reviews,avg_rating
0,948,Twilight (Twilight #1),7,3.662500
1,963,Water for Elephants,6,3.977273
2,734,The Glass Castle,6,4.206897
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.414634
4,695,The Curious Incident of the Dog in the Night-Time,6,4.081081


De lo anterior podemos identificar que algunos libros tienen muchas reseñas pero rating moderado (ej: Twilight), otros tienen rating alto con menos reseñas.

Se observa que los títulos más comentados no siempre coinciden con los mejor valorados, lo que sugiere que popularidad y satisfacción no son necesariamente equivalentes.

**3. Editorial que ha publicado el mayor numero de libros con mas de 50 paginas**

In [16]:
query = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS num_books
FROM books b
JOIN publishers p 
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY num_books DESC
LIMIT 5;
"""

top_publisher = run_query(query)
top_publisher

,publisher,num_books
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


La editorial con mayor número de libros publicados con más de 50 páginas es Penguin Books, lo que evidencia una fuerte presencia en el mercado de libros comerciales. Además, la aparición de Penguin Classics dentro del top 5 sugiere una concentración editorial relevante.

**4. Autor con la más alta calificación promedio del libro**

In [20]:
query = """
SELECT 
    a.author,
    AVG(r.rating) AS avg_rating,
    COUNT(r.rating_id) AS total_ratings
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings r ON b.book_id = r.book_id
GROUP BY a.author
HAVING COUNT(r.rating_id) >= 50
ORDER BY avg_rating DESC
LIMIT 3;
"""

result = run_query(query)
result

,author,avg_rating,total_ratings
0,Diana Gabaldon,4.300000,50
1,J.K. Rowling/Mary GrandPré,4.288462,312
2,Agatha Christie,4.283019,53


El autor con la calificacion promedio mas alta fue Diana Gabaldon con un 4.3; la sigue J.K. Rowling con un desempeño casi equivalente con un volumen significativamente mayor de calificaciones (6 veces mas).

Desde una perspectiva de negocio, esto sugiere que Rowling representa una opción más sólida y estadísticamente confiable para priorización dentro de la plataforma.

**5. Numero promedio de reseñas de texto entre usuarios que calificaron más de 50 libros**

In [24]:
query_avg_reviews = """
WITH active_users AS (
    SELECT 
        username,
        COUNT(rating_id) AS rating_count
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 50
),

user_reviews AS (
    SELECT 
        r.username,
        COUNT(rv.review_id) AS review_count
    FROM active_users r
    LEFT JOIN reviews rv ON r.username = rv.username
    GROUP BY r.username
)

SELECT 
    AVG(review_count) AS avg_reviews_per_user
FROM user_reviews;
"""

avg_reviews_result = run_query(query_avg_reviews)
avg_reviews_result


,avg_reviews_per_user
0,24.333333


El análisis muestra que los usuarios que califican más de 50 libros escriben en promedio 24.33 reseñas. Esto indica que la generación de contenido textual es significativamente menor que la actividad de calificación. Esto representa una oportunidad para incentivar la participación mediante estrategias de engagement en el nuevo producto.